In [1]:
import os
import pickle
import pandas as pd
from openai import OpenAI
import anthropic
from dotenv import load_dotenv
from tqdm import  tqdm

load_dotenv()
api_key = os.environ.get('API_KEY')
client = OpenAI(api_key=api_key)

anthropic_api_key = os.environ.get('ANTHROPIC_API_KEY')
anthropic_client = anthropic.Anthropic(
    api_key=anthropic_api_key
)

In [2]:
import random
from random import randint
import re

redacted = "[redacted]"

def scrub(text):
	text = re.sub(r"ACTRN[0-9]+", redacted, text)
	text = re.sub(r"ChiCTR[A-Z0-9-]+", redacted, text)
	text = re.sub(r"CTIS[0-9-]+", redacted, text)
	text = re.sub(r"CTRI[0-9/]+", redacted, text)
	text = re.sub(r"DRKS[0-9]+", redacted, text)
	text = re.sub(r"EUCTREUCTR[0-9a-zA-Z-]+", redacted, text)
	text = re.sub(r"IRCT[0-9]+N[0-9]+", redacted, text)
	text = re.sub(r"ISRCTN[0-9]+", redacted, text)
	text = re.sub(r"ITMCTR[0-9]+", redacted, text)
	text = re.sub(r"JPRN-[a-zA-Z0-9]+", redacted, text)
	text = re.sub(r"KCT[0-9]{7}", redacted, text)
	text = re.sub(r"LBCTR[0-9]+", redacted, text)
	text = re.sub(r"NCT[0-9]{8}", redacted, text)
	text = re.sub(r"NL-OMON[0-9]+", redacted, text)
	text = re.sub(r"PACTR[0-9]+", redacted, text)
	text = re.sub(r"RBR-[a-z0-9]+", redacted, text)
	text = re.sub(r"RPCEC[0-9]{4}", redacted, text)
	text = re.sub(r"SLCTR/\d+/\d+", redacted, text)
	text = re.sub(r"TCTR[0-9]+", redacted, text)
	return text

def discriminate(real, fake):
	real = scrub(real)
	fake = scrub(fake)
	answer = randint(0, 1)
	#print(real)
	#print(fake)
	#print(answer)
	while True:
		try:
			chat_completion = client.chat.completions.create(
				messages=[
					{
						"role": "user",
						"content": f'Which of the following abstracts is more likely to be a real abstract describing a clinical trial? Return only "1" or "2".\n\n1. "{real if answer == 0 else fake}"\n\n2. "{real if answer == 1 else fake}"',
					}
				],
				model="gpt-4o-2024-11-20",
			)
			guess = chat_completion.choices[0].message.content
			if guess == "1" or guess == "2":
				return int(guess)-1 == answer
		except Exception as e:
			pass

def anthropic_discriminate(real, fake):
	real = scrub(real)
	fake = scrub(fake)
	answer = randint(0, 1)

	while True:
		try:
			message = anthropic_client.messages.create(
				messages=[
					{
						"role": "user",
						"content": [
							{
								"type": "text",
								"text": f'Which of the following abstracts is more likely to be a real abstract describing a clinical trial? Return only "1" or "2".\n\n1. "{real if answer == 0 else fake}"\n\n2. "{real if answer == 1 else fake}"',
							}
						]
					}
				],
				model="claude-3-7-sonnet-20250219",
				max_tokens=8192,
				temperature=1
			)
			guess = message.content[0].text
			if guess == "1" or guess == "2":
				return int(guess)-1 == answer
		except Exception as e:
			pass

def evaluate_discrimination(real_abstracts, fake_abstracts, n=50, num_seeds=5, api="openai"):
    accuracy_list = []
    
    for i in range(num_seeds):
        random.seed(i)
        real_samples = random.sample(real_abstracts, n)
        fake_samples = random.sample(fake_abstracts, n)
        
        pbar = tqdm(total=n, desc=f"Seed {i+1}/{num_seeds}")
        answer_list = []
        for real, fake in zip(real_samples, fake_samples):
            if api == "openai":
                ans = discriminate(real=real, fake=fake)
            elif api == "anthropic":
                ans = anthropic_discriminate(real=real, fake=fake)
            answer_list.append(ans)
            pbar.update(1)
        pbar.close()
        
        accuracy = sum(answer_list)/len(answer_list)
        accuracy_list.append(accuracy)
        print(f"Seed {i+1} accuracy: {accuracy:.4f}")
    
    mean_accuracy = sum(accuracy_list)/len(accuracy_list)
    std_accuracy = (sum((x - mean_accuracy) ** 2 for x in accuracy_list) / len(accuracy_list)) ** 0.5
    
    print(f"Mean accuracy: {mean_accuracy:.4f} ± {std_accuracy:.4f}")
    return mean_accuracy, std_accuracy

In [25]:
anthropic_discriminate("IMPORTANCE: Cerebral palsy (CP) is the most common developmental motor disorder in children. Robot-assisted gait training (RAGT) using a wearable robot can provide intensive overground walking experience. OBJECTIVE: To investigate the effectiveness of overground RAGT in children with CP using an untethered, torque-assisted, wearable exoskeletal robot. DESIGN, SETTING, AND PARTICIPANTS: This multicenter, single-blind randomized clinical trial was conducted from September 1, 2021, to March 31, 2023, at 5 rehabilitation institutions in Korea. Ninety children with CP in Gross Motor Function Classification System levels II to IV were randomized. INTERVENTION: The RAGT group underwent 18 sessions of RAGT during 6 weeks, whereas the control group received standard physical therapy for the same number of sessions during the same period. MAIN OUTCOME AND MEASURES: The primary outcome measure was the Gross Motor Function Measure 88 (GMFM-88) score. Secondary outcome measures were the GMFM-66, Pediatric Balance Scale, selective control assessment of the lower extremity, Pediatric Evaluation of Disability Inventory-Computer Adaptive Test (PEDI-CAT), 6-minute walking test scores (distance and oxygen consumption), muscle and fat mass via bioelectrical impedance analysis, and gait parameters measured via 3-dimensional analysis. All assessments were performed for all patients at baseline, at the end of the 6-week intervention, and after the 4-week follow-up. RESULTS: Of the 90 children (mean [SD] age, 9.51 [2.48] years; 49 [54.4%] male and 41 [45.6%] female) in the study, 78 (86.7%) completed the intervention, with 37 participants (mean [SD] age, 9.57 [2.38] years; 19 [51.4%] male) and 41 participants (mean [SD] age, 9.32 [2.37] years; 26 [63.4%] male) randomly assigned to the RAGT and control groups, respectively. Changes in the RAGT group significantly exceeded changes in the control group in GMFM-88 total (mean difference, 2.64; 95% CI, 0.50-4.78), GMFM-E (mean difference, 2.70; 95% CI, 0.08-5.33), GMFM-66 (mean difference, 1.31; 95% CI, 0.01-2.60), and PEDI-CAT responsibility domain scores (mean difference, 2.52; 95% CI, 0.42-4.63), indicating independence in daily living at postintervention assessment. At the 4-week follow-up, the RAGT group showed significantly greater improvements in balance control (mean difference, 1.48; 95% CI, 0.03-2.94) and Gait Deviation Index (mean difference, 6.48; 95% CI, 2.77-10.19) compared with the control group. CONCLUSIONS AND RELEVANCE: In this randomized clinical trial, overground RAGT using a wearable robot significantly improved gross motor function and gait pattern. This new torque-assisted wearable exoskeletal robot, based on assist-as-needed control, may complement standard rehabilitation by providing adequate assistance and therapeutic support to children with CP. TRIAL REGISTRATION: CRIS Identifier: KCT0006273.",
			 "To evaluate the effectiveness of a robotic gait trainer (RTG) in improving gait function in children with cerebral palsy (CP). A randomized controlled trial was conducted in a rehabilitation hospital. Twenty children with CP (mean age 7.3 years, SD 2.1) were randomly assigned to either the RTG group (n=10) or the conventional gait training (CGT) group (n=10). The RTG group received 30 minutes of RTG training per day, 5 days a week, for 4 weeks. The CGT group received 30 minutes of CGT per day, 5 days a week, for 4 weeks. The primary outcome was the 10-meter walk test (10MWT). Secondary outcomes included the 6-minute walk test (6MWT), the Timed Up and Go test (TUG), the Berg Balance Scale (BBS), and the Gross Motor Function Measure (GMFM). The RTG group showed significant improvements in the 10MWT (p=0.001), 6MWT (p=0.001), TUG (p=0.001), and BBS (p=0.001) compared with the CGT group. The GMFM scores also showed significant improvements in the RTG group (p=0.001). The RTG group showed significant improvements in gait function compared with the CGT group. The RTG is a feasible and effective tool for improving gait function in children with CP.")

True

# Load Real Abstracts

In [3]:
with open("../data/pubmed_rct/processed_dataset/test_with_embeddings.pkl", 'rb') as f:
    test_dict = pickle.load(f)
real_abstracts = test_dict["abstracts"]

# RealEmbedding2Abstract

In [4]:
with open('../data/pubmed_rct/decoded_abstracts/5tasks_full_tuning_decoded_abstracts.pkl', 'rb') as file:
    decoded_abstracts = pickle.load(file)

In [34]:
evaluate_discrimination(real_abstracts, decoded_abstracts, n=50, num_seeds=5)

Seed 1/5: 100%|██████████| 50/50 [00:26<00:00,  1.91it/s]


Seed 1 accuracy: 0.6400


Seed 2/5: 100%|██████████| 50/50 [01:05<00:00,  1.31s/it]


Seed 2 accuracy: 0.6200


Seed 3/5: 100%|██████████| 50/50 [01:15<00:00,  1.50s/it]


...

Seed 4/5: 100%|██████████| 50/50 [01:14<00:00,  1.49s/it]


Seed 4 accuracy: 0.5000


Seed 5/5: 100%|██████████| 50/50 [01:13<00:00,  1.47s/it]

Seed 5 accuracy: 0.6000
Mean accuracy: 0.6080 ± 0.0601


(0.608, 0.06013318551349165)

In [5]:
evaluate_discrimination(real_abstracts, decoded_abstracts, n=50, num_seeds=5, api="anthropic")

Seed 1/5: 100%|██████████| 50/50 [01:10<00:00,  1.41s/it]


Seed 1 accuracy: 0.8200


Seed 2/5: 100%|██████████| 50/50 [02:07<00:00,  2.54s/it]


Seed 2 accuracy: 0.8200


Seed 3/5: 100%|██████████| 50/50 [02:09<00:00,  2.60s/it]


...

Seed 4/5: 100%|██████████| 50/50 [02:04<00:00,  2.49s/it]


Seed 4 accuracy: 0.8400


Seed 5/5: 100%|██████████| 50/50 [02:01<00:00,  2.44s/it]

Seed 5 accuracy: 0.8400
Mean accuracy: 0.8240 ± 0.0150


(0.8240000000000001, 0.014966629547095742)

# InterpolatedEmb2Abstract

In [35]:
with open('../data/pubmed_rct/decoded_abstracts/5tasks_full_tuning_interpolated_decoded_abstracts.pkl', 'rb') as file:
    interpolated_decoded_abstracts = pickle.load(file)

In [36]:
evaluate_discrimination(real_abstracts, interpolated_decoded_abstracts, n=50, num_seeds=5)

Seed 1/5: 100%|██████████| 50/50 [00:28<00:00,  1.77it/s]


Seed 1 accuracy: 0.7000


Seed 2/5: 100%|██████████| 50/50 [01:02<00:00,  1.25s/it]


Seed 2 accuracy: 0.5600


Seed 3/5: 100%|██████████| 50/50 [01:12<00:00,  1.45s/it]


...

Seed 4/5: 100%|██████████| 50/50 [01:12<00:00,  1.44s/it]


Seed 4 accuracy: 0.5400


Seed 5/5: 100%|██████████| 50/50 [01:13<00:00,  1.47s/it]

Seed 5 accuracy: 0.6600
Mean accuracy: 0.6040 ± 0.0637


(0.6040000000000001, 0.06374950980203688)

# Real versus Vec2Text (Pipeline)

In [6]:
with open('./vec2text/gtr-20steps-pipeline2abstracts.pkl', 'rb') as file:
    _, decoded_abstract_list_via_pipe = pickle.load(file)

In [21]:
evaluate_discrimination(real_abstracts, decoded_abstract_list_via_pipe, n=50, num_seeds=5)

Seed 1/5: 100%|██████████| 50/50 [00:28<00:00,  1.77it/s]


Seed 1 accuracy: 1.0000


Seed 2/5: 100%|██████████| 50/50 [00:24<00:00,  2.06it/s]


Seed 2 accuracy: 1.0000


Seed 3/5: 100%|██████████| 50/50 [00:51<00:00,  1.02s/it]


...

Seed 4/5: 100%|██████████| 50/50 [00:52<00:00,  1.06s/it]


Seed 4 accuracy: 1.0000


Seed 5/5: 100%|██████████| 50/50 [00:52<00:00,  1.06s/it]

Seed 5 accuracy: 1.0000
Mean accuracy: 1.0000 ± 0.0000


(1.0, 0.0)

In [7]:
evaluate_discrimination(real_abstracts, decoded_abstract_list_via_pipe, n=50, num_seeds=5, api="anthropic")

Seed 1/5: 100%|██████████| 50/50 [00:59<00:00,  1.19s/it]


Seed 1 accuracy: 1.0000


Seed 2/5: 100%|██████████| 50/50 [01:05<00:00,  1.32s/it]


Seed 2 accuracy: 1.0000


Seed 3/5: 100%|██████████| 50/50 [01:30<00:00,  1.82s/it]


...

Seed 4/5: 100%|██████████| 50/50 [01:29<00:00,  1.78s/it]


Seed 4 accuracy: 1.0000


Seed 5/5: 100%|██████████| 50/50 [01:27<00:00,  1.75s/it]

Seed 5 accuracy: 1.0000
Mean accuracy: 1.0000 ± 0.0000


(1.0, 0.0)

# Real versus Gender CAV (50 Male&Female cases; alpha=-0.5)

## Penalty=1.2

In [25]:
# Real versus Gender (alpha)
male_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_male_results.tsv", sep="\t")
male_cav_abstracts = male_df["abstract_-0.5"].to_list()
female_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_female_results.tsv", sep="\t")
female_cav_abstracts = female_df["abstract_0.5"].to_list()
all_gender_cav_abstracts = male_cav_abstracts + female_cav_abstracts

In [26]:
evaluate_discrimination(real_abstracts, all_gender_cav_abstracts, n=50, num_seeds=5)

Seed 1/5: 100%|██████████| 50/50 [00:25<00:00,  1.98it/s]


Seed 1 accuracy: 0.6600


Seed 2/5: 100%|██████████| 50/50 [01:07<00:00,  1.34s/it]


Seed 2 accuracy: 0.5800


Seed 3/5: 100%|██████████| 50/50 [01:16<00:00,  1.52s/it]


...

Seed 4/5: 100%|██████████| 50/50 [01:12<00:00,  1.46s/it]


Seed 4 accuracy: 0.5400


Seed 5/5: 100%|██████████| 50/50 [01:14<00:00,  1.48s/it]

Seed 5 accuracy: 0.5800
Mean accuracy: 0.6200 ± 0.0716


(0.62, 0.07155417527999328)

# Penalty=1.0

In [29]:
# Real versus Gender (alpha)
male_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.0/all_male_results.tsv", sep="\t")
male_cav_abstracts = male_df["abstract_-0.5"].to_list()
female_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.0/all_female_results.tsv", sep="\t")
female_cav_abstracts = female_df["abstract_0.5"].to_list()
all_gender_cav_abstracts = male_cav_abstracts + female_cav_abstracts

In [30]:
evaluate_discrimination(real_abstracts, all_gender_cav_abstracts, n=50, num_seeds=5)

Seed 1/5: 100%|██████████| 50/50 [00:37<00:00,  1.33it/s]


Seed 1 accuracy: 0.7800


Seed 2/5: 100%|██████████| 50/50 [01:34<00:00,  1.88s/it]


Seed 2 accuracy: 0.7400


Seed 3/5: 100%|██████████| 50/50 [01:41<00:00,  2.04s/it]


...

Seed 4/5: 100%|██████████| 50/50 [01:27<00:00,  1.74s/it]


Seed 4 accuracy: 0.6400


Seed 5/5: 100%|██████████| 50/50 [01:34<00:00,  1.89s/it]

Seed 5 accuracy: 0.6800
Mean accuracy: 0.7120 ± 0.0483


(0.7120000000000001, 0.04833218389437828)

# Real versus Age CAV (50 Aged&Child cases; alpha=-0.5)

## Penalty = 1.2

In [27]:
aged_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_aged_results.tsv", sep="\t")
aged_cav_abstracts = aged_df["abstract_-0.5"].to_list()
child_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.2/all_child_results.tsv", sep="\t")
child_cav_abstracts = child_df["abstract_0.5"].to_list()
all_age_cav_abstracts = aged_cav_abstracts + child_cav_abstracts

In [28]:
evaluate_discrimination(real_abstracts, all_age_cav_abstracts, n=50, num_seeds=5)

Seed 1/5: 100%|██████████| 50/50 [00:52<00:00,  1.05s/it]


Seed 1 accuracy: 0.7000


Seed 2/5: 100%|██████████| 50/50 [01:14<00:00,  1.49s/it]


Seed 2 accuracy: 0.5800


Seed 3/5: 100%|██████████| 50/50 [01:14<00:00,  1.49s/it]


...

Seed 4/5: 100%|██████████| 50/50 [01:13<00:00,  1.48s/it]


Seed 4 accuracy: 0.5200


Seed 5/5: 100%|██████████| 50/50 [01:12<00:00,  1.45s/it]

Seed 5 accuracy: 0.5200
Mean accuracy: 0.6000 ± 0.0769


(0.6, 0.07694153624668537)

## Penalty = 1.0

In [31]:
aged_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.0/all_aged_results.tsv", sep="\t")
aged_cav_abstracts = aged_df["abstract_-0.5"].to_list()
child_df = pd.read_csv("../data/pubmed_rct/cav_files/penalty=1.0/all_child_results.tsv", sep="\t")
child_cav_abstracts = child_df["abstract_0.5"].to_list()
all_age_cav_abstracts = aged_cav_abstracts + child_cav_abstracts

In [32]:
evaluate_discrimination(real_abstracts, all_age_cav_abstracts, n=50, num_seeds=5)

Seed 1/5: 100%|██████████| 50/50 [00:28<00:00,  1.78it/s]


Seed 1 accuracy: 0.6800


Seed 2/5: 100%|██████████| 50/50 [01:26<00:00,  1.73s/it]


Seed 2 accuracy: 0.6400


Seed 3/5: 100%|██████████| 50/50 [01:33<00:00,  1.87s/it]


...

Seed 4/5: 100%|██████████| 50/50 [01:28<00:00,  1.78s/it]


Seed 4 accuracy: 0.5600


Seed 5/5: 100%|██████████| 50/50 [01:27<00:00,  1.76s/it]

Seed 5 accuracy: 0.5600
Mean accuracy: 0.6240 ± 0.0543


(0.624, 0.054258639865002144)